# Tuzatilgan tashqi datasetni ko‘rish

Foydalanuvchi so‘rovi bilan faqat ikki CSV saqlandi: `benchmark_all_samples.csv` va `benchmark_standardized_samples.csv`. Har birida barcha 2613 namuna bor. Eski nusxalar va yordamchi CSVlar o‘chirildi.

`text` — oldingi bosqichda imlosi tuzatilgan transcript. `standardized_text` — alohida adabiylashtirilgan variant; audio bilan tasdiqlanmagan. `included_in_filtered` oldingi filtrlangan to‘plamga kirganini bildiradi.

## 1. Fayllarni ochish


In [1]:
from pathlib import Path
import csv
import subprocess
import sys
import json
PROJECT = Path.cwd().resolve()
DATA = PROJECT / "external_validation/Benchmark/karakalpak_hf_dataset"
with (DATA / "benchmark_standardized_samples.csv").open(encoding="utf-8-sig", newline="") as handle:
    samples = list(csv.DictReader(handle))
print("Jami namunalar:", len(samples))
print("Adabiy varianti farq qilgan:", sum(r["text"] != r["standardized_text"] for r in samples))


Jami namunalar: 2613
Adabiy varianti farq qilgan: 74


## 2. Audio va ikkita CSV mosligini tekshirish

Bu tekshiruv audio tinglanganini anglatmaydi. Qo‘shimcha CSV yaratilmaydi.


In [2]:
result = subprocess.run([sys.executable, str(PROJECT / "scripts/audit_external_transcripts.py"), "--check"],
                        cwd=PROJECT, capture_output=True, text=True, check=True)
summary = json.loads(result.stdout)
print("Tekshirilgan audiolar:", summary["audio_files_checked"])
print("O‘qish xatolari:", summary["audio_read_errors"])


Tekshirilgan audiolar: 2613
O‘qish xatolari: 0


## 3. Matnni ko‘rish va audioni tinglash

`show_sample(0)` birinchi namuna audiosini chiqaradi. Indeks 0–2612 oralig‘ida. `play=False` faqat matnni ko‘rsatadi. Jupyter/IPython audio player uchun kerak; asosiy tekshiruv standart Python bilan ishlaydi.


In [3]:
def show_sample(index, play=True):
    row = samples[index]
    print("Audio:", row["audio_path"])
    print("Transcript:", row["text"])
    print("Adabiy variant:", row["standardized_text"])
    if play:
        from IPython.display import Audio, display
        display(Audio(filename=str(DATA / row["audio_path"])))

show_sample(0, play=False)


Audio: audio/hf_audio_000001.wav
Transcript: kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur
Adabiy variant: kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur


## Holat

Notebookning kod celllari oddiy Python sifatida bajarildi. Jupyter ko‘rinishi va audio playback bu muhitda tekshirilmagan. To‘liq audio–matn mosligi hali tasdiqlanmagan. Benchmark WER/CER uchun adabiy sinonimlar bilan almashtirilgan matnni avtomatik reference sifatida ishlatmang.
